# Corrected focal + Dice adversarial perturbations (v2)

This notebook generates a new, versioned attack dataset. It uses target-class focal + soft-Dice for local patch optimization, fixed-subset convergence diagnostics, cosine PGD decay, best-checkpoint selection, and a pinned AnomalyCLIP checkout.

Before running: enable a Kaggle GPU and Internet, and attach only the dataset(s) selected in `DATASETS`. Run one scope and normally one dataset per Kaggle session for full generation. Start with `SMOKE_TEST=True`; after it passes, set it to `False` and pin `REPOSITORY_REF` to the commit containing this notebook.

In [ ]:
# User settings
REPOSITORY_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
REPOSITORY_REF = 'main'  # For publication, replace with the exact corrected commit SHA.
DATASETS = ('mvtec',)    # Choose: ('mvtec',), ('visa',), or ('mvtec', 'visa')
SCOPES = ('dataset',)     # Choose from: dataset, category, image
SMOKE_TEST = True         # Set False only after the diagnostic smoke run passes.
ATTACK_TRAIN_FRACTION = '1.0'
GPU = '0'


In [ ]:
# Runtime and dataset discovery
import os, subprocess, sys
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator and restart the session.')
print('GPU:', torch.cuda.get_device_name(0))

def shallow_input_directories():
    root = Path('/kaggle/input')
    for top in root.iterdir():
        if top.is_dir():
            yield top
            for child in top.iterdir():
                if child.is_dir():
                    yield child

def find_dataset_root(candidates, predicate, label):
    checked = set()
    for candidate in [*map(Path, candidates), *shallow_input_directories()]:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.add(candidate)
        if candidate.is_dir() and predicate(candidate):
            print(f'{label}: {candidate}')
            return candidate
    raise FileNotFoundError(f'{label} root was not found. Attach the dataset or add its mounted path to the candidates.')

valid_datasets = {'mvtec', 'visa'}
if not DATASETS or len(set(DATASETS)) != len(DATASETS) or not set(DATASETS) <= valid_datasets:
    raise ValueError("DATASETS must be ('mvtec',), ('visa',), or ('mvtec', 'visa')")
MVTEC_ROOT = find_dataset_root([
    '/kaggle/input/mvtec-ad/mvtec_anomaly_detection',
    '/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection',
], lambda root: (root / 'bottle' / 'test').is_dir() and (root / 'bottle' / 'train' / 'good').is_dir(), 'MVTec') if 'mvtec' in DATASETS else Path('/kaggle/working/unused_mvtec')
VISA_ROOT = find_dataset_root([
    '/kaggle/input/visa-ad/VisA_20220922',
    '/kaggle/input/visa/VisA_20220922',
    '/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922',
], lambda root: (root / 'split_csv' / '1cls.csv').is_file() or (root / '1cls.csv').is_file(), 'VisA') if 'visa' in DATASETS else Path('/kaggle/working/unused_visa')


In [ ]:
# Fetch exactly the requested generator revision
WORKING = Path('/kaggle/working')
REPO_ROOT = WORKING / 'adversarial-robustness'
if not (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--depth', '1', 'origin', REPOSITORY_REF], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', '--force', 'FETCH_HEAD'], check=True)
RESOLVED_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Generator commit:', RESOLVED_COMMIT)
if REPOSITORY_REF == 'main':
    print('WARNING: pin REPOSITORY_REF to', RESOLVED_COMMIT, 'before producing the publishable full dataset.')


In [ ]:
# Configure and run. Smoke and full outputs are deliberately separated.
dataset_tag = '-'.join(DATASETS)
OUTPUT_BASE = WORKING / (f'canonical_clip_v2_smoke_{dataset_tag}' if SMOKE_TEST else f'canonical_clip_v2_full_{dataset_tag}')
env = os.environ.copy()
env.update({
    'MVTEC_ROOT': str(MVTEC_ROOT),
    'VISA_ROOT': str(VISA_ROOT),
    'OUTPUT_BASE': str(OUTPUT_BASE),
    'GPU': GPU,
    'PYTHON_BIN': sys.executable,
    'USE_VENV': 'false',
    'PER_DATASET_BATCH_SIZE': '1',
    'ATTACK_TRAIN_FRACTION': ATTACK_TRAIN_FRACTION,
    'GENERATION_DATASETS': ','.join(DATASETS),
    'RUN_PER_DATASET': str('dataset' in SCOPES).lower(),
    'RUN_PER_CATEGORY': str('category' in SCOPES).lower(),
    'RUN_PER_IMAGE': str('image' in SCOPES).lower(),
})
if SMOKE_TEST:
    env.update({
        'ATTACK_TRAIN_FRACTION': '0.05',
        'PER_DATASET_STEPS': '8',
        'PER_CATEGORY_STEPS': '8',
        'PER_IMAGE_STEPS': '3',
        'PER_IMAGE_EVALUATION_FRACTION': '0.02',
        'DIAGNOSTIC_INTERVAL': '2',
        'DIAGNOSTIC_MAX_SAMPLES': '4',
    })
GENERATOR_DIR = REPO_ROOT / 'perturbation_generation'
subprocess.run(['bash', str(GENERATOR_DIR / 'train.sh')], cwd=GENERATOR_DIR, env=env, check=True)
print('Output:', OUTPUT_BASE)


In [ ]:
# Mandatory convergence and packaging audit
import pandas as pd
diagnostic_files = sorted(OUTPUT_BASE.rglob('optimization_diagnostics.csv'))
if not diagnostic_files:
    raise RuntimeError('No optimization_diagnostics.csv was produced.')
frames = []
for path in diagnostic_files:
    frame = pd.read_csv(path)
    frame.insert(0, 'diagnostics_file', str(path.relative_to(OUTPUT_BASE)))
    frames.append(frame)
diagnostics = pd.concat(frames, ignore_index=True)
display(diagnostics.groupby(['scope', 'loss_mode'])[['initial_total_loss', 'final_total_loss', 'total_loss_reduction']].mean())
passed = diagnostics['convergence_check_passed'].astype(str).str.lower().eq('true')
failed = diagnostics[~passed]
if len(failed):
    display(failed)
    raise RuntimeError(f'{len(failed)} conditions failed the fixed-objective convergence check; do not publish these artifacts.')
archives = sorted(OUTPUT_BASE.glob('*segmentation_loss_v2.zip'))
if not archives:
    raise RuntimeError('No v2 ZIP archive was produced.')
for archive in archives:
    print(f'{archive.name}: {archive.stat().st_size / 2**30:.3f} GiB')
print('AUDIT PASSED. Publish only the non-smoke archives as a new Kaggle dataset version.')
